# Convert loaded epochs to MNE

This notebook keeps the loading path deliberately simple: choose one condition, load a small set of epochs with `TrialHandler`, then wrap the returned data and metadata in an `mne.EpochsArray`.

No datastore inspection is needed here; this is the shape of the workflow that can later become a loader option.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import mne
import numpy as np
import pandas as pd

from tnsd_access import TrialHandler

mne.set_log_level("WARNING")
pd.set_option("display.max_columns", 50)

## 1. Load one condition

`DATA_ROOT` should point at the dataset folder prepared in `getting_started.ipynb`. Edit `CONDITION` to any NSD condition/stimulus ID you want to inspect.

In [ ]:
DATA_ROOT = Path("temporal-natural-scenes-dataset")
VERSION = "v1"

SUBJECT = 1
CONDITION = 5
N_TRIALS = 8
CHANNELS = ["A1", "A2", "A3", "A4"]
TMIN = -0.1
TMAX = 0.8

loader = TrialHandler(DATA_ROOT, version=VERSION)
trials = loader.lookup_trials(subject=SUBJECT, condition=CONDITION).head(N_TRIALS)

if trials.empty:
    raise ValueError("No trials matched this subject/condition. Try another CONDITION from the metadata.")

trials[["subject", "session", "run", "epoch", "condition", "trial_type"]]

In [ ]:
result = loader.get_data(
    trials,
    channels=CHANNELS,
    tmin=TMIN,
    tmax=TMAX,
)

data = result["data"]
metadata = result["metadata"].copy().reset_index(drop=True)

print("data shape:", data.shape)
metadata.head()

## 2. Make an MNE `EpochsArray`

The loaded array has shape `(n_epochs, n_channels, n_times)`. MNE also needs an `Info` object and an events array. Here the event code comes from the `condition` column in the returned metadata.

For this notebook we infer the sampling rate from the requested crop window and the number of returned samples. Once this lives in the loader, the loader can provide the exact time vector directly.

In [ ]:
n_epochs, n_channels, n_times = data.shape
sfreq = n_times / (TMAX - TMIN)

info = mne.create_info(
    ch_names=CHANNELS,
    sfreq=sfreq,
    ch_types="eeg",
)

event_codes = metadata["condition"].astype(int).to_numpy()
events = np.column_stack(
    [
        np.arange(n_epochs, dtype=int),
        np.zeros(n_epochs, dtype=int),
        event_codes,
    ]
)
event_id = {f"condition/{code}": int(code) for code in np.unique(event_codes)}

epochs = mne.EpochsArray(
    data,
    info,
    events=events,
    tmin=TMIN,
    event_id=event_id,
    metadata=metadata,
    baseline=None,
    verbose=False,
)

epochs

## 3. Set a montage

The selected channel names are standard 10-20 labels, so MNE can place them with the built-in `standard_1020` montage.

In [ ]:
montage = mne.channels.make_standard_montage("standard_1020")
epochs.set_montage(montage, match_case=False, on_missing="warn")

epochs.info

## 4. Plot

In [ ]:
epochs.plot(
    n_epochs=min(5, len(epochs)),
    n_channels=len(CHANNELS),
    scalings="auto",
);

In [ ]:
evoked = epochs.average()
evoked.plot(spatial_colors=True);

In [ ]:
epochs.plot_image(picks=CHANNELS[0], combine=None);

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(epochs.times, evoked.data.T)
ax.axvline(0, color="black", linewidth=1, alpha=0.6)
ax.set_xlabel("Time (s)")
ax.set_ylabel("Amplitude")
ax.set_title(f"Condition {CONDITION}: average across loaded trials")
ax.legend(CHANNELS, loc="upper right", frameon=False)
plt.show()

## 5. Notebook helper

This small helper captures the conversion above without changing the loader yet.

In [ ]:
def result_to_mne_epochs(result, channel_names, tmin, tmax, montage_name="standard_1020"):
    data = result["data"]
    metadata = result["metadata"].copy().reset_index(drop=True)
    n_epochs, _, n_times = data.shape
    sfreq = n_times / (tmax - tmin)

    info = mne.create_info(channel_names, sfreq=sfreq, ch_types="eeg")
    event_codes = metadata["condition"].astype(int).to_numpy()
    events = np.column_stack(
        [np.arange(n_epochs, dtype=int), np.zeros(n_epochs, dtype=int), event_codes]
    )
    event_id = {f"condition/{code}": int(code) for code in np.unique(event_codes)}

    epochs = mne.EpochsArray(
        data,
        info,
        events=events,
        tmin=tmin,
        event_id=event_id,
        metadata=metadata,
        baseline=None,
        verbose=False,
    )
    epochs.set_montage(
        mne.channels.make_standard_montage(montage_name),
        match_case=False,
        on_missing="warn",
    )
    return epochs


epochs_again = result_to_mne_epochs(result, CHANNELS, TMIN, TMAX)
epochs_again